In [ ]:

# PROJETO DE ZOONOSES
# CONSTRUCAO DO BANCO DE DADOS SQLITE
# VERSAO 1.0

# 1. IMPORTACAO DAS BIBLIOTECAS

import sqlite3
import os
from datetime import datetime

# 2. CONFIGURACAO DO PROJETO — SEM GOOGLE DRIVE

# O banco novo fica no armazenamento temporario do Colab.
# Ele e independente do banco antigo e nao e salvo no Google Drive.
PASTA_PROJETO = "/content/PROJETO_ZOONOSES"
os.makedirs(PASTA_PROJETO, exist_ok=True)

CAMINHO_BANCO = os.path.join(
    PASTA_PROJETO,
    "banco_zoonoses_v2.db"
)

# 3. CONEXAO COM SQLITE

conexao = sqlite3.connect(CAMINHO_BANCO)

cursor = conexao.cursor()

# Ativar integridade dos relacionamentos
cursor.execute("PRAGMA foreign_keys = ON")

# 4. CRIACAO DAS TABELAS

cursor.executescript("""

-- ----------------------------------------------------------
-- TABELA 1: LOTES DE DADOS
-- Registra cada geração ou importação de dados.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS lotes (

    id_lote INTEGER PRIMARY KEY AUTOINCREMENT,

    tipo_lote TEXT NOT NULL,

    quantidade_informada INTEGER,

    quantidade_gerada INTEGER,

    data_criacao TEXT NOT NULL,

    descricao TEXT,

    status TEXT DEFAULT 'Registrado'

);


-- ----------------------------------------------------------
-- TABELA 2: REGISTROS BRUTOS
-- Preserva os dados originais, inclusive os problemáticos.
-- Os dados são armazenados em formato JSON.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS registros_brutos (

    id_registro INTEGER PRIMARY KEY AUTOINCREMENT,

    id_lote INTEGER,

    conjunto TEXT NOT NULL,

    dados_json TEXT NOT NULL,

    data_recebimento TEXT NOT NULL,

    FOREIGN KEY (id_lote)
        REFERENCES lotes(id_lote)

);


-- ----------------------------------------------------------
-- TABELA 3: ANIMAIS
-- Armazena os animais após validação e tratamento.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS animais (

    id_animal TEXT PRIMARY KEY,

    nome TEXT,

    especie TEXT,

    raca TEXT,

    sexo TEXT,

    municipio TEXT,

    peso_kg REAL,

    status TEXT,

    vacinas_faltantes TEXT,

    situacao_vacinacao TEXT,

    imagem TEXT,

    revisao_peso TEXT,

    data_cadastro TEXT,

    id_lote INTEGER,

    origem TEXT DEFAULT 'Sintetica',

    FOREIGN KEY (id_lote)
        REFERENCES lotes(id_lote)

);


-- ----------------------------------------------------------
-- TABELA 4: VACINACOES
-- Um animal pode possuir varias vacinacoes.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS vacinacoes (

    id_vacinacao INTEGER PRIMARY KEY AUTOINCREMENT,

    id_animal TEXT NOT NULL,

    nome_vacina TEXT NOT NULL,

    data_aplicacao TEXT,

    proxima_dose TEXT,

    situacao TEXT,

    observacoes TEXT,

    FOREIGN KEY (id_animal)
        REFERENCES animais(id_animal)

);


-- ----------------------------------------------------------
-- TABELA 5: DIAGNOSTICOS
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS diagnosticos (

    id_diagnostico INTEGER PRIMARY KEY AUTOINCREMENT,

    id_animal TEXT NOT NULL,

    data_diagnostico TEXT,

    diagnostico TEXT,

    descricao TEXT,

    situacao TEXT,

    origem TEXT DEFAULT 'Sintetica',

    FOREIGN KEY (id_animal)
        REFERENCES animais(id_animal)

);


-- ----------------------------------------------------------
-- TABELA 6: MAUS-TRATOS
-- Permite registrar ocorrencias ainda sem animal identificado.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS maus_tratos (

    id_ocorrencia INTEGER PRIMARY KEY AUTOINCREMENT,

    id_animal TEXT,

    municipio TEXT,

    data_ocorrencia TEXT,

    tipo_ocorrencia TEXT,

    descricao TEXT,

    situacao TEXT,

    origem TEXT DEFAULT 'Sintetica',

    FOREIGN KEY (id_animal)
        REFERENCES animais(id_animal)

);


-- ----------------------------------------------------------
-- TABELA 7: OBITOS E EUTANASIAS
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS obitos_eutanasias (

    id_evento INTEGER PRIMARY KEY AUTOINCREMENT,

    id_animal TEXT,

    tipo_evento TEXT NOT NULL,

    data_evento TEXT,

    municipio TEXT,

    motivo TEXT,

    observacoes TEXT,

    origem TEXT DEFAULT 'Sintetica',

    FOREIGN KEY (id_animal)
        REFERENCES animais(id_animal)

);


-- ----------------------------------------------------------
-- TABELA 8: AUDITORIA
-- Registra inclusoes, atualizacoes e correcoes.
-- ----------------------------------------------------------

CREATE TABLE IF NOT EXISTS auditoria (

    id_auditoria INTEGER PRIMARY KEY AUTOINCREMENT,

    entidade TEXT NOT NULL,

    id_registro TEXT,

    operacao TEXT NOT NULL,

    descricao TEXT,

    data_operacao TEXT NOT NULL

);


-- ----------------------------------------------------------
-- INDICES PARA MELHORAR AS CONSULTAS
-- ----------------------------------------------------------

CREATE INDEX IF NOT EXISTS idx_animais_municipio
ON animais(municipio);

CREATE INDEX IF NOT EXISTS idx_animais_especie
ON animais(especie);

CREATE INDEX IF NOT EXISTS idx_diagnosticos_animal
ON diagnosticos(id_animal);

CREATE INDEX IF NOT EXISTS idx_maus_tratos_animal
ON maus_tratos(id_animal);

CREATE INDEX IF NOT EXISTS idx_brutos_lote
ON registros_brutos(id_lote);

CREATE INDEX IF NOT EXISTS idx_brutos_conjunto
ON registros_brutos(conjunto);

""")

# 5. IDENTIFICACAO DA VERSAO DO BANCO

cursor.execute("PRAGMA user_version = 1")

# 6. SALVAR AS ALTERACOES

conexao.commit()

# 7. VERIFICAR AS TABELAS CRIADAS

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""")

tabelas = cursor.fetchall()

print("\n" + "=" * 55)
print("BANCO DE DADOS DE ZOONOSES")
print("=" * 55)

print("\nTABELAS CRIADAS:")

for tabela in tabelas:
    print(" -", tabela[0])

# 8. VERIFICAR QUANTIDADE DE ANIMAIS

cursor.execute("SELECT COUNT(*) FROM animais")

total_animais = cursor.fetchone()[0]

print("\nTOTAL DE ANIMAIS CADASTRADOS:", total_animais)

print("\nLOCAL DO BANCO:")
print(CAMINHO_BANCO)

print("\nESTRUTURA CRIADA E SALVA COM SUCESSO!")

# 9. ENCERRAR CONEXAO

conexao.close()


In [ ]:

import sqlite3

# Reutiliza CAMINHO_BANCO definido na primeira celula.

conexao = sqlite3.connect(CAMINHO_BANCO)
cursor = conexao.cursor()

try:
    # Verificar as colunas existentes na tabela animais
    cursor.execute("PRAGMA table_info(animais)")
    colunas = [coluna[1] for coluna in cursor.fetchall()]

    # Adicionar somente as colunas que ainda não existem
    novas_colunas = {
        "data_acolhimento": "TEXT",
        "motivo_acolhimento": "TEXT",
        "origem_acolhimento": "TEXT"
    }

    for nome, tipo in novas_colunas.items():
        if nome not in colunas:
            cursor.execute(
                f"ALTER TABLE animais ADD COLUMN {nome} {tipo}"
            )
            print(f"Coluna adicionada: {nome}")
        else:
            print(f"Coluna já existente: {nome}")

    # Criar controle permanente de sequências
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS controle_sequencias (
            nome TEXT PRIMARY KEY,
            proximo_valor INTEGER NOT NULL
        )
    """)

    cursor.execute("""
        INSERT OR IGNORE INTO controle_sequencias
        (nome, proximo_valor)
        VALUES ('animais', 1)
    """)

    conexao.commit()

    print("\nAtualização concluída!")

except Exception as erro:
    conexao.rollback()
    print("Erro:", erro)

finally:
    conexao.close()
